### Semantic Chunking
- SemanticChunker is a document splitter that uses embedding similarity between sentences to decide chunk boundaries.

- It ensures that each chunk is semantically coherent and not cut off mid-thought like traditional character/token splitters.

In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

c:\Users\sumsaras\Documents\PersonalWorkspace\AI_Learning\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [23]:
model = SentenceTransformer("all-MiniLM-L6-v2")
text = """
LangChain is a framework for building applications with LLMs.
LangChain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

# Step 1. Splitting into multiple chunks
sentences = [s.strip() for s in text.split("\n") if s.strip()]

# Step 2. Embed each sentence
embeddings = model.encode(sentences)
# print(embeddings)

# Step 3. Initialize params
threshold = 0.7
chunks = []
current_chunk = [sentences[0]]

# Step 4. Cosine Similarity/ Semantic grouping based on threshold
for i in range(1, len(sentences)):
    similarity = cosine_similarity([embeddings[i - 1]], [embeddings[i]])[0][0]
    if similarity > threshold:
        current_chunk.append(sentences[i])
    else:
        chunks.append(" ".join(current_chunk))
        current_chunk = [sentences[i]]

chunks.append(" ".join(current_chunk))


for index, chunk in enumerate(chunks):
    print(f"\n Chunk {index + 1} : \n{chunk}")


 Chunk 1 : 
LangChain is a framework for building applications with LLMs. LangChain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.

 Chunk 2 : 
You can create chains, agents, memory, and retrievers.

 Chunk 3 : 
The Eiffel Tower is located in Paris.

 Chunk 4 : 
France is a popular tourist destination.


### Modular Class Coding

In [20]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.documents import Document
from langchain_astradb import AstraDBVectorStore
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableLambda, RunnableMap
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from typing import List

class SemanticChunker:
    def __init__(self, model="all-MiniLM-L6-v2", threshold=0.7):
        self.model = SentenceTransformer(model)
        self.threshold = threshold
    
    def semantic_splitter(self, text : str) -> List[str]:
        sentences = [s.strip() for s in text.split(".") if s.strip()]
        chunks = []
        current_chunk = [sentences[0]]
        embeddings = self.model.encode(sentences)
        for i in range(1, len(sentences)):
            semantic_similarity = cosine_similarity([embeddings[i - 1]], [embeddings[i]])[0][0]
            if semantic_similarity > self.threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(". ".join(current_chunk))
                # chunks.append(current_chunk)
                current_chunk = [sentences[i]]

        chunks.append(". ".join(current_chunk))
        return chunks

    def split_documents(self, docs : List[Document]) -> List[Document]:
        documents = []
        for doc in docs:
            for chunk in self.semantic_splitter(doc.page_content):
                documents.append(Document(metadata=doc.metadata, page_content=chunk))
        return documents



In [22]:
text = """
LangChain is a framework for building applications with LLMs.
LangChain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""
chunker = SemanticChunker()
documents = chunker.split_documents([Document(page_content=text)])


In [53]:
from langchain_astradb import AstraDBVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model = "sentence-transformers/all-MiniLM-L6-v2")
vector_store_db = AstraDBVectorStore(
    embedding=embeddings,
    api_endpoint= "https://d23e5536-e548-48a9-b764-25e8fcbafd39-us-east-2.apps.astra.datastax.com",
    token="AstraCS:PLTQMkUzNwpeiMorexqDbrJz:73692a8977a5b42de0b221d55e10c90a1fda728740ad682798c18d2b7f102965",
    collection_name="advanced_chunking"
)  

vector_store_db.add_documents(documents)

['2cb76df6c9c649cfa447d88767c20fbf',
 '191a237cc23241b2a9f5f7c3c87ef3bd',
 '61b246e5f5db4dceb439ab9065a3b00d',
 'cb8422bc3e3f44aab95f84359add611d']

In [54]:
retriever = vector_store_db.as_retriever()


In [55]:
prompt = """
Answer the asked question based on the following context

Context : {context}

Question : {question}
"""

promptTemplate = PromptTemplate.from_template(prompt)
promptTemplate

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nAnswer the asked question based on the following context\n\nContext : {context}\n\nQuestion : {question}\n')

In [56]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [57]:
llm = init_chat_model(model='openai/gpt-oss-120b',temperature=0.4, model_provider='groq')
rag_chain = (
    RunnableMap(
        {
            "context" : lambda x:retriever.invoke(x["question"]),
            "question": lambda x: x["question"]
        }
    )
    | promptTemplate
    | llm
    | StrOutputParser()
)
rag_chain

{
  context: RunnableLambda(...),
  question: RunnableLambda(...)
}
| PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nAnswer the asked question based on the following context\n\nContext : {context}\n\nQuestion : {question}\n')
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000029A83D44AF0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000029A83D45370>, model_name='openai/gpt-oss-120b', temperature=0.4, model_kwargs={}, groq_api_key=SecretStr('**********'))
| StrOutputParser()

In [62]:
result = rag_chain.invoke({"question":"What is lanchain learning"})
result

'**LangChain learning** refers to learning about the **LangChain** framework—a set‑of modular abstractions designed to help developers build applications that use large language models (LLMs).  \n\n- LangChain lets you combine LLMs with external tools (e.g., OpenAI, Pinecone).  \n- It provides building blocks such as **chains**, **agents**, **memory**, and **retrievers** that you can assemble to create sophisticated LLM‑driven workflows.  \n\nSo, “lanchain learning” is essentially studying how to use LangChain to construct and manage LLM‑based applications.'

### Semantic Chunker using langchain lib

In [65]:
from langchain_openai import OpenAIEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.document_loaders import TextLoader

In [ ]:
## Load the documents
loader=TextLoader("langchain_intro.txt")
docs=loader.load()

## Initialize embedding model
embedding=HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")

## Create the semantic chunker
chunker=SemanticChunker(embedding)

## Split the documents
chunks=chunker.split_documents(docs)

## Result

for i,chunk in enumerate(chunks):
    print(f"\n chunk {i+1}:\n{chunk.page_content}")


 chunk 1:
LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone. You can create chains, agents, memory, and retrievers. The Eiffel Tower is located in Paris.

 chunk 2:
France is a popular tourist destination.
